In [ ]:
import cv2
import numpy as np

images_names = ['low_light.png', 'outside.png']

# Reduce contrast and add noise to the images

In [ ]:

for image_name in images_names:
  
  image = cv2.imread(f'./images/{image_name}')

  # reduce contrast around the mean
  alpha = 0.6
  gray = np.full_like(image, int(image.mean()))
  low = cv2.addWeighted(image, alpha, gray, 1 - alpha, 0)

  # Add gaussian noise
  mean= 0
  std = 25
  noise = np.random.normal(mean, std, low.shape)
  noisy_image = np.clip(low.astype(np.float32) + noise, 0, 255).astype(np.uint8)
      

  cv2.imwrite(f'./images/degraded/{image_name}', noisy_image)

# Linear contrast strecthing

In [38]:

for image_name in images_names:
  image_filepath = f'./images/degraded/{image_name}'
  
  # Read the image and convert it to HLS
  image = cv2.imread(image_filepath)
  image_hls = cv2.cvtColor(image, cv2.COLOR_BGR2HLS)
  
  # RGB VERSION
  # Separate the channels
  b, g, r = cv2.split(image)
  b_f = b.astype(np.float32)
  g_f = g.astype(np.float32)
  r_f = r.astype(np.float32)
  
  # Run linear contrast stretching formula on each channel 
  # ((v - v_min)/(v_max - v_min)) * 255 where v is the value of the channel
  new_r = ((r_f - r_f.min()) / (r_f.max() - r_f.min()) * 255).astype(np.uint8)
  new_g = ((g_f - g_f.min()) / (g_f.max() - g_f.min()) * 255).astype(np.uint8)
  new_b = ((b_f - b_f.min()) / (b_f.max() - b_f.min()) * 255).astype(np.uint8)
  
  # Merge the channels back
  out_image_rgb = cv2.merge([new_b,new_g,new_r])
  cv2.imwrite(f'./images/output/linear_stretch/rgb_{image_name}', out_image_rgb)
  
  # LUMINANCE VERSION
  # Separate the channels
  h, l, s = cv2.split(image_hls)
  
  # Convert to float
  l_f = l.astype(np.float32)
  
  # Run linear contrast stretching formula on the luminance 
  # ((l - l_min)/(l_max - l_min)) * 255
  new_l = ((l_f - l_f.min()) / (l_f.max() - l_f.min()) * 255).astype(np.uint8)
  
  # Merge the channels back
  out_image_hls = cv2.cvtColor(cv2.merge([h,new_l,s]), cv2.COLOR_HLS2BGR)
  
  # Save the image so that we can calculate metrics later
  cv2.imwrite(f'./images/output/linear_stretch/hls_{image_name}', out_image_hls)

# Histogram equalization

In [ ]:
for image_name in images_names:
  image_filepath = f'./images/degraded/{image_name}'
  
  # Read the image and convert it to HSV
  image = cv2.imread(image_filepath)
  image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  
  # Split the channels
  h, s, v = cv2.split(image_hsv)
  
  # Equalize the value channel
  new_v = cv2.equalizeHist(v)
  
  # Merge the channels and convert back to bgr
  out_image_hsv = cv2.cvtColor(cv2.merge([h,s,new_v]), cv2.COLOR_HSV2BGR)
  
  # Save the image so that we can calculate metrics later
  cv2.imwrite(f'./images/output/histogram_equalization/{image_name}', out_image_hsv)
  

# Denoising + Gamma correction

In [48]:
for image_name in images_names:
  image_filepath = f'./images/degraded/{image_name}'
  
  # Read the image
  image = cv2.imread(image_filepath)
  
  # Apply bilater filter
  image_smooth = cv2.bilateralFilter(image, 9, 75, 75)
  
  # Up the gamma  

  # Create a LUT as there are only 256 possible values
  # Gamma correction: gamma < 1 brightens
  gamma = 0.7
  lut = np.array([255 * (i / 255) ** gamma for i in range(256)]).clip(0, 255).astype(np.uint8)
  
  # Each value is given directly by the LUT
  image_gamma = cv2.LUT(image_smooth, lut)
  
  cv2.imwrite(f'./images/output/smoothing/{image_name}', image_gamma)